# Día 7 — Pipelines, forecasting y comunicación

Cerrar el flujo reproducible y presentar una conclusión útil, honesta y verificable.

## Objetivos de aprendizaje

- Automatizar transformaciones y modelo en un Pipeline.
- Crear lags sin usar información futura.
- Separar Train/Validation/Test cronológicamente y comparar baselines.
- Comunicar evidencia, limitaciones y una solución reproducible.

> **Tiempo del plan oficial:** 2 horas sincrónicas + 1 hora autónoma. Prioriza pipeline, baseline temporal y presentación; deja extensiones de horizonte y diagnóstico adicional para el trabajo autónomo.

**Dataset:** `datos/macro_mensual_demo.csv`, serie sintética didáctica explícitamente no oficial. Se carga desde disco; no se consulta ningún servicio externo.

## Cómo estudiar esta jornada

Lee primero la pregunta y la explicación; ejecuta luego las celdas en orden. Después de cada tabla o gráfico, detente en **Interpreta** y redacta una conclusión con evidencia. Los resultados dependen de la versión de las librerías y de las particiones; usa tus propios números y no copies los ejemplos como verdades universales. No confundas predicción con causalidad.

## 1. Qué significa automatizar un análisis

Un Pipeline reúne transformaciones y estimador para que cada ajuste aprenda solo de Train. Aplicaremos ese principio a un flujo de pronóstico de un paso adelante: predecir el índice mensual de actividad con su historia reciente y variables macroeconómicas observadas. No mezclaremos pasado y futuro al validar.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def buscar_datos():
    candidatos = [Path("../datos"), Path("datos"), Path.cwd() / "datos", Path.cwd().parent / "datos"]
    for ruta in candidatos:
        if ruta.exists():
            return ruta.resolve()
    raise FileNotFoundError("No encuentro la carpeta datos/. Abre Jupyter desde la raíz o notebooks/ del repositorio.")

DATA_DIR = buscar_datos()
print("Datos locales:", DATA_DIR)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

serie = pd.read_csv(DATA_DIR / "macro_mensual_demo.csv", parse_dates=["fecha"])
serie = serie.sort_values("fecha").reset_index(drop=True)
serie.head()


### Procedencia y límites de estos datos

`macro_mensual_demo.csv` es una serie **sintética didáctica** creada para practicar flujo temporal; no son cifras oficiales, no describen la economía ecuatoriana real ni sirven para decisiones. La usamos porque 144 meses permiten demostrar lags y evaluación temporal en una clase breve. Un proyecto aplicado debe sustituirla por una serie pública local descargada, versionada y documentada, nunca conectarse a una API durante la clase.

In [ ]:
print("Dimensiones:", serie.shape)
print("Rango:", serie.fecha.min(), "a", serie.fecha.max())
print("Duplicados de fecha:", serie.fecha.duplicated().sum())
print("Nulos:")
print(serie.isna().sum())
plt.figure(figsize=(10,4))
plt.plot(serie.fecha, serie.indice_actividad)
plt.title("Índice mensual de actividad — serie sintética didáctica")
plt.ylabel("Índice (unidad didáctica)"); plt.xlabel("Fecha"); plt.grid(alpha=.2)


## 2. Crear predictores que solo miran al pasado

`lag_1` es el valor del mes anterior; `lag_12`, el de doce meses antes. La media móvil se desplaza antes de calcularse para que incluya meses pasados, no el target del mismo mes. Este cuidado previene leakage temporal.

In [ ]:
s = serie.copy()
s["lag_1"] = s.indice_actividad.shift(1)
s["lag_3"] = s.indice_actividad.shift(3)
s["lag_12"] = s.indice_actividad.shift(12)
s["media_3_previa"] = s.indice_actividad.shift(1).rolling(3).mean()
s["mes"] = s.fecha.dt.month
s["anio"] = s.fecha.dt.year
s = s.dropna().reset_index(drop=True)
features = ["precio_petroleo_usd", "inflacion_interanual_pct", "exportaciones_mill",
            "lag_1", "lag_3", "lag_12", "media_3_previa", "mes", "anio"]
target = "indice_actividad"
print("Filas después de lags:", len(s))
s[["fecha", target, "lag_1", "lag_12", "media_3_previa"]].head()


### Verifica una fila a mano

El target de una fecha solo utiliza observaciones anteriores para crear sus lags. Verifica que `lag_1` corresponda al mes inmediatamente previo y que las primeras doce filas desaparezcan por falta de historial. En forecasting las variables externas también deben estar disponibles en la fecha de pronóstico; en este ejemplo sintético asumimos que sí, pero en operación esa disponibilidad requiere análisis de publicación y revisiones.

## 3. Separación temporal y baseline

Reservamos el 20 % final como Test, manteniendo el orden. Para seleccionar configuraciones, dividimos el periodo previo de forma cronológica en Train y Validation. Un baseline `lag_1` dice que el siguiente mes será igual al último observado; un baseline estacional usa `lag_12`. Son referencias obligatorias.

In [ ]:
corte = int(len(s) * .80)
corte_val = int(len(dev) * .80)
train, val = dev.iloc[:corte_val].copy(), dev.iloc[corte_val:].copy()
print("Train:", train.fecha.min().date(), "–", train.fecha.max().date(), len(train))
print("Validation:", val.fecha.min().date(), "–", val.fecha.max().date(), len(val))
print("Test:", test.fecha.min().date(), "–", test.fecha.max().date(), len(test))


In [ ]:
def mae(y, p): return mean_absolute_error(y, p)
for nombre, col in [("Persistencia lag 1", "lag_1"), ("Estacional lag 12", "lag_12")]:
    print(nombre, "MAE Validation:", round(mae(val[target], val[col]), 3))


### Interpreta el baseline temporal

Compara los dos MAE y considera el patrón mensual. Menor MAE es mejor en la unidad sintética del índice. El baseline estacional requiere doce meses de historia; el de persistencia puede responder mejor ante series suaves. Ninguno usa covariables externas: un modelo más complejo debe superar una referencia razonable fuera de muestra.

## 4. Pipeline y selección pequeña de modelos

Las covariables aquí son numéricas. El Pipeline incluye imputación mediana, escalado y un regresor; Random Forest no necesita escalado, pero lo mantenemos para ilustrar el flujo automatizado reutilizable. Ajustaremos pocas configuraciones dentro del desarrollo y elegiremos con Validation, no con Test.

In [ ]:
pipe = Pipeline([
    ("prep", ColumnTransformer([("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                                                     ("scale", StandardScaler())]), features)])),
    ("model", RandomForestRegressor(random_state=42, n_estimators=200, min_samples_leaf=3))
])
pipe.fit(train[features], train[target])
pred_val = pipe.predict(val[features])
print("MAE RF Validation:", round(mae(val[target], pred_val), 3))
print("MAE baseline lag1:", round(mae(val[target], val.lag_1), 3))
print("MAE baseline lag12:", round(mae(val[target], val.lag_12), 3))


### Diagnóstico de Validation

¿El Pipeline supera ambos baselines? Un resultado peor no es un fracaso: evita justificar complejidad innecesaria. Inspecciona errores por fecha y revisa si hay cambios de régimen. Con 144 observaciones y una serie sintética, los errores y la selección son ilustrativos; no son una evaluación económica formal.

## 5. Decisión final y Test

Antes de ejecutar, declara una regla simple: elegir Random Forest solo si mejora el menor MAE de los baselines en Validation; de otro modo elegir el baseline ganador. Fijamos la regla y evaluamos una sola vez en el tramo final.

In [ ]:
mae_rf_val = mae(val[target], pred_val)
mae_lag1_val = mae(val[target], val.lag_1)
mae_lag12_val = mae(val[target], val.lag_12)
elegido = "rf" if mae_rf_val < min(mae_lag1_val, mae_lag12_val) else ("lag_1" if mae_lag1_val <= mae_lag12_val else "lag_12")
if elegido == "rf":
    pred_test = pipe.predict(test[features])
else:
    pred_test = test[elegido].to_numpy()
pd.DataFrame([{"modelo":elegido, "MAE_validation":min(mae_rf_val,mae_lag1_val,mae_lag12_val),
               "MAE_test":mae(test[target],pred_test),
               "RMSE_test":mean_squared_error(test[target],pred_test)**.5}]).set_index("modelo").round(3)


### Comunica el pronóstico

MAE es el desvío absoluto promedio en unidades del índice; RMSE penaliza más los errores extremos. Compara ambos con Validation, mira fechas donde el modelo falla y explica si el método superó baselines. Si el test final es peor, puede indicar variación temporal; no reajustes mirando repetidamente ese periodo. La decisión automática del ejemplo hace explícita la regla de selección, no una garantía de óptimo.

## 6. Presentación de resultados y proyecto integrador

Una presentación de 5–7 minutos responde: problema/decisión; fuente, licencia, periodo y unidad de observación; calidad y disponibilidad temporal; baseline; separación temporal; Pipeline; métricas en unidades comprensibles; diagnóstico de residuos/errores; límites, revisiones, drift y siguiente paso. Para el BCE, evitar datos confidenciales y distinguir una asociación predictiva de una recomendación de política.

### Lista de verificación del proyecto final (40 puntos sugeridos)

- **Problema y fuente (5):** pregunta útil, procedencia, licencia, variables y unidad claros.
- **Datos y preparación (5):** EDA, calidad, faltantes, transformaciones reproducibles y leakage controlado.
- **Baseline/modelos (10):** al menos dos alternativas pertinentes y Pipeline reproducible.
- **Evaluación (10):** split apropiado, métricas justificadas, comparación y Test reservado.
- **Interpretación y límites (5):** resultado en unidades del dominio, errores y cautela causal.
- **Comunicación (5):** visualización legible, síntesis ejecutiva y reproducibilidad.

El instructor puede ajustar esta rúbrica a la evaluación institucional; los 40 puntos corresponden al componente sumativo del plan de clase.

## Actividad autónoma

Cambia la regla de selección para optimizar una métrica distinta o define un horizonte de dos meses, creando los targets sin que los lags incluyan futuro. Explica cómo cambian los baselines, el tamaño de la muestra y la interpretación. No presentes datos sintéticos como datos del BCE.